# Deep Model Training — R(2+1)D-18, CNN-GRU & TSM (Final / v2)

Final dissertation-quality training of all three deep video architectures for the ES327 driver-drowsiness study. Trains under identical subject-independent conditions and saves the best checkpoint per model to `outputs_all_3_models_v2/`.

- **R(2+1)D-18** — 3D-ResNet (best internal/UTA-RLDD F1)
- **CNN-GRU** — ResNet-18 per-frame → GRU (hidden=128, dropout=0.3; best external/DROZY robustness)
- **TSM-fast** — ResNet-18 + temporal averaging (lightweight baseline)

**Training features:** class-weighted loss, AMP mixed precision, ReduceLROnPlateau, optional backbone freeze for CNN-GRU, robust NPZ loading with corrupt-clip filtering, early stopping on validation F1.

> These v2 checkpoints are the final ones used for all evaluation, results, and Grad-CAM analysis.

---
## 1. Setup & Paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import pandas as pd
from collections import Counter
from pathlib import Path

SPLITS_DIR   = "/content/drive/MyDrive/ES327_Drowsiness/splits"
CLIPS_DRIVE  = "/content/drive/MyDrive/ES327_Drowsiness/Clips"
CLIPS_LOCAL  = "/content/Clips"

SAVE_ROOT = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models"  # where your best.pt live
print("SAVE_ROOT:", SAVE_ROOT)

In [ ]:
import torch
NUM_CLASSES = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| NUM_CLASSES:", NUM_CLASSES)

---
## 2. Load Splits & Cache Clips Locally

Load and combine the UTA-RLDD + YawDD train/val splits, drop a known-corrupt clip, and rsync the required clips to Colab's local SSD for fast loading.

In [ ]:
import os
import pandas as pd
from collections import Counter

SPLITS_DIR = "/content/drive/MyDrive/ES327_Drowsiness/splits"

UTA_TRAIN  = os.path.join(SPLITS_DIR, "uta_train.csv")
UTA_VAL    = os.path.join(SPLITS_DIR, "uta_val.csv")
UTA_TEST   = os.path.join(SPLITS_DIR, "uta_test.csv")
YAW_TRAIN  = os.path.join(SPLITS_DIR, "yawdd_train.csv")
YAW_VAL    = os.path.join(SPLITS_DIR, "yawdd_val.csv")
DROZY_TEST = os.path.join(SPLITS_DIR, "drozy_test.csv")

def load_split_csv(path):
    df = pd.read_csv(path)
    df = df.dropna(subset=["clip_path", "label"]).copy()
    df["label"] = df["label"].astype(int)
    return df

def namespace_subjects(df, default_ds):
    df = df.copy()
    if "dataset" in df.columns:
        ds = df["dataset"].fillna(default_ds).astype(str).str.lower()
    else:
        ds = pd.Series([default_ds]*len(df), index=df.index)
    df["subject_id"] = ds + "_" + df["subject_id"].astype(str)
    return df

# Load
uta_train  = namespace_subjects(load_split_csv(UTA_TRAIN),  "uta")
uta_val    = namespace_subjects(load_split_csv(UTA_VAL),    "uta")
uta_test   = namespace_subjects(load_split_csv(UTA_TEST),   "uta")
yaw_train  = namespace_subjects(load_split_csv(YAW_TRAIN),  "yawdd")
yaw_val    = namespace_subjects(load_split_csv(YAW_VAL),    "yawdd")
drozy_test = namespace_subjects(load_split_csv(DROZY_TEST), "drozy")

# Combine
train_df    = pd.concat([uta_train, yaw_train], ignore_index=True)
val_df      = pd.concat([uta_val,   yaw_val],   ignore_index=True)
test_df     = uta_test.copy()
external_df = drozy_test.copy()

print("Train:", len(train_df), dict(Counter(train_df["label"])))
print("Val:", len(val_df), dict(Counter(val_df["label"])))
print("Test:", len(test_df), dict(Counter(test_df["label"])))
print("External:", len(external_df), dict(Counter(external_df["label"])))

In [ ]:
import os
import pandas as pd

CLIPS_DRIVE = "/content/drive/MyDrive/ES327_Drowsiness/Clips"
CLIPS_LOCAL = "/content/Clips"

# train_df + val_df already loaded above (must have clip_path)
all_paths = pd.concat([train_df["clip_path"], val_df["clip_path"]]).dropna().astype(str).unique()

prefix = CLIPS_DRIVE.rstrip("/") + "/"
rel_paths = [p[len(prefix):] for p in all_paths if p.startswith(prefix)]

print("Train+Val unique clips:", len(rel_paths))

filelist = "/content/needed_trainval.txt"
with open(filelist, "w") as f:
    f.write("\n".join(rel_paths))

print("✅ Wrote:", filelist)
print("Example:", rel_paths[0])

In [ ]:
filelist = "/content/needed_trainval.txt"
bad_sub = "Fold3_part1__29__10__00122.npz"  # match by substring

with open(filelist, "r") as f:
    lines = [ln.strip() for ln in f if ln.strip()]

kept = [ln for ln in lines if bad_sub not in ln]

with open(filelist, "w") as f:
    f.write("\n".join(kept) + "\n")

print("List size before:", len(lines))
print("List size after :", len(kept))
print("Removed         :", len(lines) - len(kept))

In [ ]:
!mkdir -p /content/Clips

!rsync -ah \
  --info=progress2 \
  --partial \
  --ignore-existing \
  --ignore-missing-args \
  --ignore-errors \
  --files-from=/content/needed_trainval.txt \
  "/content/drive/MyDrive/ES327_Drowsiness/Clips/" \
  "/content/Clips/"

In [ ]:
CLIPS_DRIVE_P = Path(CLIPS_DRIVE)
CLIPS_LOCAL_P = Path(CLIPS_LOCAL)

def map_to_local_df(df):
    df = df.copy()
    def _map(p):
        p = str(p)
        pref = str(CLIPS_DRIVE_P) + "/"
        if p.startswith(pref):
            rel = p[len(pref):]
            local = CLIPS_LOCAL_P / rel
            if local.exists():
                return str(local)
        return p
    df["clip_path_mapped"] = df["clip_path"].astype(str).apply(_map)
    return df

train_df_m    = map_to_local_df(train_df)
val_df_m      = map_to_local_df(val_df)
test_df_m     = map_to_local_df(test_df)
external_df_m = map_to_local_df(external_df)

print("Mapped counts:",
      "train", len(train_df_m),
      "val", len(val_df_m),
      "test", len(test_df_m),
      "ext", len(external_df_m))

print("Example mapped path:", train_df_m["clip_path_mapped"].iloc[0])

In [ ]:
train_df_m    = map_to_local_df(train_df)
val_df_m      = map_to_local_df(val_df)
test_df_m     = map_to_local_df(test_df)
external_df_m = map_to_local_df(external_df)

print("Mapped OK.")

---
## 3. Model Builders

Defines all three architectures (3D-ResNet-18, CNN-GRU, TSM-fast).

In [ ]:
import torchvision
import torch.nn as nn

NUM_CLASSES = 2

# 3D-CNN: 3D ResNet-18
def build_r3d18(num_classes=2):
    m = torchvision.models.video.r3d_18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

# CNN-GRU: ResNet18 per-frame + GRU
class ResNetGRU(nn.Module):
    def __init__(self, hidden=256, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.feat_dim = 512
        self.gru = nn.GRU(self.feat_dim, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, num_classes)

    def forward(self, x):  # (B,T,C,H,W)
        B,T,C,H,W = x.shape
        x = x.reshape(B*T, C, H, W)
        f = self.backbone(x).flatten(1)
        f = f.reshape(B, T, self.feat_dim)
        out, _ = self.gru(f)
        return self.fc(out[:, -1, :])

def build_cnn_gru(num_classes=2):
    return ResNetGRU(hidden=256, num_classes=num_classes)

# TSM-style temporal average baseline
class ResNetTemporalAvg(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):  # (B,C,T,H,W)
        B,C,T,H,W = x.shape
        x = x.permute(0,2,1,3,4).contiguous().view(B*T, C, H, W)
        f = self.backbone(x).flatten(1).view(B, T, 512).mean(dim=1)
        return self.fc(f)

def build_tsm_fast(num_classes=2):
    return ResNetTemporalAvg(num_classes=num_classes)

print("✅ Builders defined.")

---
## 4. Train All Three Models (v2)

Self-contained dissertation-quality training cell. Redefines the improved CNN-GRU (hidden=128, dropout=0.3), builds robust loaders, and trains TSM-fast, CNN-GRU and R(2+1)D-18 in sequence — saving `best.pt`, `bad_files.txt` and `epoch_metrics.csv` per model under `outputs_all_3_models_v2/`.

In [ ]:
# ============================================================
# DISSERTATION-QUALITY TRAINING PIPELINE (3 MODELS)
# - Robust NPZ loader w/ bad-clip filtering
# - Class-weighted loss (handles imbalance; improves recall/F1)
# - AMP mixed precision (fast on T4)
# - ReduceLROnPlateau scheduler (stabilises; avoids collapse)
# - CNN-GRU improved (hidden=128 + dropout=0.3)
# - Optional 2-epoch backbone freeze for CNN-GRU (enabled by default)
# - Per-model sensible batch sizes + patience
# - Logs: best.pt, bad_files.txt, epoch_metrics.csv
# ============================================================

import os, numpy as np, torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, confusion_matrix
)

# -------------------------
# Globals
# -------------------------
SAVE_ROOT = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2"
os.makedirs(SAVE_ROOT, exist_ok=True)

# expects: device, NUM_CLASSES, train_df_m, val_df_m
# and build_tsm_fast, build_r3d18 (we redefine build_cnn_gru below)

assert "device" in globals(), "device not defined"
assert "NUM_CLASSES" in globals(), "NUM_CLASSES not defined"
assert "train_df_m" in globals() and "val_df_m" in globals(), "train_df_m / val_df_m not defined"

print("✅ device:", device)
print("✅ SAVE_ROOT:", SAVE_ROOT)

# ============================================================
# 0) CNN-GRU (improved) builder (hidden=128 + dropout=0.3)
#    + backbone freeze for first N epochs (default: 2)
# ============================================================
import torchvision

class ResNetGRU(nn.Module):
    def __init__(self, hidden=128, dropout=0.30, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])  # (B*T,512,1,1)
        self.feat_dim = 512
        self.gru = nn.GRU(self.feat_dim, hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, num_classes)

    def forward(self, x):  # (B,T,C,H,W)
        B, T, C, H, W = x.shape
        x = x.reshape(B * T, C, H, W)
        f = self.backbone(x).flatten(1)              # (B*T,512)
        f = f.reshape(B, T, self.feat_dim)           # (B,T,512)
        out, _ = self.gru(f)                         # (B,T,hidden)
        out_last = self.drop(out[:, -1, :])          # (B,hidden)
        return self.fc(out_last)

def build_cnn_gru(num_classes=2):
    return ResNetGRU(hidden=128, dropout=0.30, num_classes=num_classes)

print("✅ CNN-GRU builder updated (hidden=128, dropout=0.30)")

# ============================================================
# 1) Dataset that flags bad files AND returns path
# ============================================================
class NPZClipsDataset(Dataset):
    def __init__(self, df, mode="cthw", clip_t=16, clip_h=112, clip_w=112, clip_c=3):
        self.df = df.reset_index(drop=True)
        self.mode = mode
        self.clip_t = clip_t
        self.clip_h = clip_h
        self.clip_w = clip_w
        self.clip_c = clip_c

    def __len__(self):
        return len(self.df)

    def _dummy_cthw(self):
        return np.zeros((self.clip_c, self.clip_t, self.clip_h, self.clip_w), dtype=np.float32)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row["clip_path_mapped"]
        y = int(row["label"])

        bad = 0
        try:
            z = np.load(path, allow_pickle=False)
            x = z["x"] if "x" in z.files else z[z.files[0]]
            x = x.astype(np.float32)

            if x.ndim != 4:
                raise ValueError(f"bad ndim={x.ndim}")

            # Normalize to (C,T,H,W)
            if x.shape[-1] in (1, 3):          # (T,H,W,C)
                x = np.transpose(x, (3, 0, 1, 2))
            elif x.shape[1] in (1, 3):         # (T,C,H,W)
                x = np.transpose(x, (1, 0, 2, 3))
            elif x.shape[0] in (1, 3):         # already (C,T,H,W)
                pass
            else:
                raise ValueError(f"unrecognized layout shape={x.shape}")

            expected = (self.clip_c, self.clip_t, self.clip_h, self.clip_w)
            if x.shape != expected:
                raise ValueError(f"wrong shape {x.shape} expected {expected}")

            if not np.isfinite(x).all():
                raise ValueError("nan/inf found")

        except Exception:
            bad = 1
            x = self._dummy_cthw()

        if self.mode == "cthw":
            xt = torch.from_numpy(x)  # (C,T,H,W)
        elif self.mode == "tchw":
            xt = torch.from_numpy(np.transpose(x, (1, 0, 2, 3)))  # (T,C,H,W)
        else:
            raise ValueError("mode must be cthw or tchw")

        return (
            xt,
            torch.tensor(y, dtype=torch.long),
            torch.tensor(bad, dtype=torch.bool),
            path
        )

def make_loaders(train_df, val_df, mode, batch_size, num_workers=2):
    train_ds = NPZClipsDataset(train_df, mode=mode)
    val_ds   = NPZClipsDataset(val_df,   mode=mode)

    # Colab-safe: do NOT keep workers persistent
    persistent = False  # <- key change

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda"),
        persistent_workers=persistent
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda"),
        persistent_workers=persistent
    )
    return train_loader, val_loader

# ============================================================
# 2) Helpers to handle "bad" samples
# ============================================================
def _filter_bad_batch(x, y, bad, paths):
    bad = bad.bool()
    n_bad = int(bad.sum().item())
    if n_bad == 0:
        return x, y, 0, []
    bad_idx = bad.nonzero(as_tuple=False).view(-1).tolist()
    bad_paths = [paths[i] for i in bad_idx]
    good_mask = ~bad
    if int(good_mask.sum().item()) == 0:
        return None, None, n_bad, bad_paths
    return x[good_mask], y[good_mask], n_bad, bad_paths

# ============================================================
# 3) Evaluation (skips bad samples)
# ============================================================
def evaluate(model, loader):
    model.eval()
    ys, probs, preds = [], [], []
    bad_total = 0

    with torch.no_grad():
        for x, y, bad, paths in loader:
            x, y, n_bad, _ = _filter_bad_batch(x, y, bad, paths)
            bad_total += n_bad
            if x is None:
                continue

            x = x.to(device, non_blocking=True).float() / 255.0
            y = y.to(device, non_blocking=True)

            logits = model(x)
            p = torch.softmax(logits, dim=1)[:, 1]
            pred = torch.argmax(logits, dim=1)

            ys.append(y.cpu().numpy())
            probs.append(p.cpu().numpy())
            preds.append(pred.cpu().numpy())

    if len(ys) == 0:
        return {
            "acc": 0.0, "prec": 0.0, "rec": 0.0, "f1": 0.0, "prauc": 0.0,
            "cm": np.array([[0, 0], [0, 0]]), "bad_total": bad_total
        }

    y = np.concatenate(ys)
    pr = np.concatenate(preds)
    pb = np.concatenate(probs)

    return {
        "acc": accuracy_score(y, pr),
        "prec": precision_score(y, pr, zero_division=0),
        "rec": recall_score(y, pr, zero_division=0),
        "f1": f1_score(y, pr, zero_division=0),
        "prauc": average_precision_score(y, pb),
        "cm": confusion_matrix(y, pr),
        "bad_total": bad_total
    }

# ============================================================
# 4) Train (class-weighted loss + scheduler + AMP)
#    + CNN-GRU: freeze backbone first 2 epochs (best chance)
# ============================================================
def train_one(
    name, builder, mode, batch_size,
    epochs=20, lr=1e-4, num_workers=4,
    patience=5, log_every=200,
    freeze_backbone_epochs=0
):
    exp_dir = os.path.join(SAVE_ROOT, name)
    os.makedirs(exp_dir, exist_ok=True)

    best_path = os.path.join(exp_dir, "best.pt")
    bad_log_path = os.path.join(exp_dir, "bad_files.txt")
    epoch_log_path = os.path.join(exp_dir, "epoch_metrics.csv")

    train_loader, val_loader = make_loaders(
        train_df_m, val_df_m,
        mode=mode, batch_size=batch_size, num_workers=num_workers
    )

    model = builder(NUM_CLASSES).to(device)
    opt = optim.AdamW(model.parameters(), lr=lr)

    # ---- class-weighted loss (most important single improvement) ----
    pos = int((train_df_m["label"] == 1).sum())
    neg = int((train_df_m["label"] == 0).sum())
    w_pos = neg / max(1, pos)
    class_weights = torch.tensor([1.0, float(w_pos)], device=device)
    crit = nn.CrossEntropyLoss(weight=class_weights)
    print(f"[{name}] class_weights={class_weights.tolist()} (pos={pos}, neg={neg})")

    # ---- scheduler: stabilises + avoids collapse ----
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=1)

    # ---- AMP for speed on CUDA ----
    use_amp = (device.type == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    # clear logs
    with open(bad_log_path, "w") as f:
        f.write("")
    with open(epoch_log_path, "w") as f:
        f.write("epoch,lr,train_loss,val_f1,val_acc,val_prauc,bad_train,bad_val\n")

    best_f1 = -1.0
    bad_epochs = 0

    # helper to freeze/unfreeze backbone (CNN-GRU)
    def _set_backbone_trainable(trainable: bool):
        for module_name in ["backbone"]:
            if hasattr(model, module_name):
                for p in getattr(model, module_name).parameters():
                    p.requires_grad = trainable

    for ep in range(1, epochs + 1):
        # ---- optional freeze backbone early for CNN-GRU stability ----
        if freeze_backbone_epochs > 0:
            if ep <= freeze_backbone_epochs:
                _set_backbone_trainable(False)
            elif ep == freeze_backbone_epochs + 1:
                _set_backbone_trainable(True)
                # re-create optimizer so unfrozen params get updates
                opt = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=float(opt.param_groups[0]["lr"]))
                sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=1)
                print(f"[{name}] ✅ unfroze backbone at epoch {ep}")

        model.train()
        losses = []
        bad_seen = 0
        bad_paths_epoch = []
        running_loss = 0.0

        for step, (x, y, bad, paths) in enumerate(train_loader, start=1):
            x, y, n_bad, bad_paths = _filter_bad_batch(x, y, bad, paths)
            bad_seen += n_bad
            if bad_paths:
                bad_paths_epoch.extend(bad_paths)
            if x is None:
                continue

            x = x.to(device, non_blocking=True).float() / 255.0
            y = y.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(x)
                loss = crit(logits, y)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            li = float(loss.item())
            losses.append(li)
            running_loss += li

            if log_every is not None and step % log_every == 0:
                avg_so_far = running_loss / max(1, len(losses))
                print(f"[{name}] ep{ep}/{epochs} step {step}/{len(train_loader)} loss~{avg_so_far:.4f} (bad {bad_seen})")

        # log bad files (dedup)
        if bad_paths_epoch:
            bad_paths_epoch = sorted(set(p for p in bad_paths_epoch if p))
            with open(bad_log_path, "a") as f:
                for p in bad_paths_epoch:
                    f.write(p + "\n")

        # eval
        m = evaluate(model, val_loader)
        avg_loss = float(np.mean(losses)) if losses else 0.0

        # scheduler step on val f1
        sched.step(m["f1"])
        cur_lr = float(opt.param_groups[0]["lr"])

        print(
            f"[{name}] ep{ep}/{epochs} "
            f"lr={cur_lr:.2e} train_loss={avg_loss:.4f} "
            f"val_f1={m['f1']:.4f} val_acc={m['acc']:.4f} val_prauc={m['prauc']:.4f} "
            f"bad_train={bad_seen} bad_val={m.get('bad_total',0)}"
        )

        with open(epoch_log_path, "a") as f:
            f.write(f"{ep},{cur_lr},{avg_loss},{m['f1']},{m['acc']},{m['prauc']},{bad_seen},{m.get('bad_total',0)}\n")

        # early stop + save best
        if m["f1"] > best_f1:
            best_f1 = m["f1"]
            bad_epochs = 0
            torch.save({
                "name": name,
                "epoch": ep,
                "model_state": model.state_dict(),
                "optimizer_state": opt.state_dict(),
                "best_f1": best_f1,
            }, best_path)
            print(f"✅ saved best -> {best_path}")
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"⏹️ early stopping at ep{ep} (best f1={best_f1:.4f})")
                break

    del model
    torch.cuda.empty_cache()
    return best_path, best_f1, bad_log_path, epoch_log_path

# ============================================================
# 5) Run all models (good default settings for T4)
# ============================================================
RUNS = [
    ("tsm_fast", build_tsm_fast, "cthw", 32, 0),  # freeze epochs = 0
    ("cnn_gru",  build_cnn_gru,  "tchw", 16, 2),  # freeze backbone first 2 epochs
    ("r3d18",    build_r3d18,    "cthw", 12, 0),  # freeze epochs = 0
]

results = {}
for name, builder, mode, bs, freeze_ep in tqdm(RUNS, desc="Models", position=0):
    # a bit tighter patience for cnn_gru
    this_patience = 3 if name == "cnn_gru" else 5

    best_path, best_f1, bad_log, epoch_log = train_one(
        name, builder, mode, bs,
        epochs=20, lr=1e-4,
        num_workers=4,
        patience=this_patience,
        log_every=200,
        freeze_backbone_epochs=freeze_ep
    )
    results[name] = {
        "best_f1": best_f1,
        "ckpt": best_path,
        "bad_log": bad_log,
        "epoch_log": epoch_log
    }

print("\n=== DONE ===")
results